In [12]:
import sqlite3
from pathlib import Path
import pandas as pd

db_path = Path("/Users/chowingchan/Desktop/Project/AI-Analytics-Copilot/Competitive-Intelligence-Internal-Analytics-System/data/database/thelook_ecommerce.db")
conn = sqlite3.connect(db_path)
cursor = conn.cursor()

In [13]:
#Schema validation
pd.read_sql("""
SELECT name, type
FROM sqlite_master
ORDER BY type, name;
""", conn)

,name,type
0,distribution_centers,table
1,events,table
2,inventory_items,table
3,mart_user_segment,table
4,order_items,table
5,orders,table
6,products,table
7,users,table
8,mart_daily_sales,view
9,mart_order_summary,view


In [17]:
#Row count validation
import pandas as pd

sql_query = """
SELECT 
    (SELECT COUNT(*) FROM mart_order_summary) AS order_summary_cnt,
    (SELECT COUNT(*) FROM mart_daily_sales) AS daily_sales_cnt,
    (SELECT COUNT(*) FROM mart_product_sales) AS product_sales_cnt,
    (SELECT COUNT(*) FROM mart_user_segment) AS user_segment_cnt;
"""

df = pd.read_sql(sql_query, conn)
print(df)

   order_summary_cnt  daily_sales_cnt  product_sales_cnt  user_segment_cnt
0             124923             1215              44940            100000


In [21]:
#Grain validation
# should be 1 row per order / per date
 
sql_query = """
SELECT
    (SELECT SUM(cnt) FROM(SELECT order_id, COUNT(*) AS cnt FROM mart_order_summary GROUP BY order_id HAVING COUNT(*)>1) AS t1) AS order_summary_dup_cnt,
    (SELECT SUM(cnt) FROM(SELECT order_date, COUNT(*) AS cnt FROM mart_daily_sales GROUP BY order_date HAVING COUNT(*)>1) AS t2) AS order_summary_dup_cnt;
"""

df = pd.read_sql(sql_query, conn)
print(df)


  order_summary_dup_cnt order_summary_dup_cnt
0                  None                  None


In [23]:
# Mart vs Raw Table

sql_query = """
SELECT 
    (SELECT SUM(order_revenue) FROM mart_order_summary WHERE status = 'Complete') AS order_summary_revenue,
    (SELECT SUM(oi.sale_price) FROM orders o JOIN order_items oi ON o.order_id = oi.order_id WHERE o.status = 'Complete') AS user_segment_cnt;
"""

df = pd.read_sql(sql_query, conn)
print(df)

   order_summary_revenue  user_segment_cnt
0           2.674091e+06      2.674091e+06


In [24]:
#Null / abnormal value check
pd.read_sql("""
SELECT *
FROM mart_order_summary
WHERE order_revenue < 0 OR order_cost < 0 OR gross_profit IS NULL;
""", conn)

,order_id,user_id,status,order_created_at,order_returned_at,order_shipped_at,order_delivered_at,item_count,order_revenue,order_cost,gross_profit,gross_margin


In [25]:
# Date logic check
pd.read_sql("""
SELECT MIN(order_created_at), MAX(order_created_at) FROM mart_order_summary;
""", conn)

,MIN(order_created_at),MAX(order_created_at)
0,2019-01-12 12:17:00+00:00,2022-06-08 18:37:00+00:00
